In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("mohammad2012191/q1-stage-3-2026")

print("Path to dataset files:", path)

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
from torchvision.datasets import ImageFolder
from torchvision import transforms
import random

from torch.utils.data import random_split
from torch.utils.data import DataLoader
import torch
import torch.nn as nn

import torch.optim as optim

In [ ]:
# Write your code here

# Just inspected the data, it's split into train and test, and each has classes as file names
# There must be a catch, I don't trust whoever wrote the exam


train_path = os.path.join(path,'PlantVillage','train')
test_path = os.path.join(path,'PlantVillage' ,'test')
# Anyways, I'll create two imagefolder objects, one for train and one for test
# Define transforms
train_transform = transforms.Compose([
    transforms.Resize((32, 32)),
    # transforms.RandomHorizontalFlip(0.5),
    transforms.RandomRotation(15),
    transforms.ToTensor()
])


test_transform = transforms.Compose([
    transforms.Resize((32, 32)),
    transforms.ToTensor()
])

train_dataset = ImageFolder(root =train_path, transform = train_transform)
test_dataset = ImageFolder(root =test_path, transform = test_transform)

# Create DataLoaders
# (The Dataset Class loads only one sample at a time. We pass it to dataloader to read batch_size of images at a time)
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, num_workers=2) # num_workers specifies the number of subprocesses to use for parallel data loading
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False, num_workers=2)

# Check dataset size
print(f"Training samples: {len(train_dataset)}, Testing samples: {len(test_dataset)}")

In [ ]:
fig, axes = plt.subplots(2, 6, figsize=(10, 4))
for ax in axes.ravel():
    idx = np.random.randint(0, len(train_dataset))
    sample = train_dataset[idx]
    x, y = sample

    # unnormalize for display (approx)
    x_vis = (x * 0.5) + 0.5
    x_vis = x_vis.permute(1, 2, 0)
    ax.imshow(x_vis.squeeze(0), cmap="gray")
    ax.set_title(train_dataset.classes[y], fontsize=9)
    ax.axis("off")
plt.tight_layout()
plt.show()

In [ ]:
class SimpleCNN(nn.Module):
  def __init__(self, num_classes = 3):
    super().__init__()# We use this to extract features to feed to the FC classifiier
    self.features = nn.Sequential(
        nn.Conv2d(in_channels=3, out_channels=16, kernel_size=3, padding = 1), # padding = 1 means adding zeros around the image, if we want info about the edges use different padding
        nn.BatchNorm2d(16),
        nn.ReLU(),
        nn.MaxPool2d(kernel_size=2), # If I don't specify the stride, just assume it's the same as the kernel size, good info to plug in the formula

        nn.Conv2d(in_channels=16, out_channels=32, kernel_size=3, padding = 1),
        nn.BatchNorm2d(32),
        nn.ReLU(),
        nn.MaxPool2d(kernel_size=2),

        nn.Conv2d(in_channels=32, out_channels=64, kernel_size=3, padding = 1),
        nn.BatchNorm2d(64),
        nn.ReLU(),
        nn.MaxPool2d(kernel_size=2),

        nn.Conv2d(in_channels=64, out_channels=128, kernel_size=3, padding = 1),
        nn.BatchNorm2d(128),
        nn.ReLU(),
        nn.MaxPool2d(kernel_size=2),

        nn.Conv2d(in_channels=128, out_channels=256, kernel_size=3, padding = 1),
        nn.BatchNorm2d(256),
        nn.ReLU(),
        nn.MaxPool2d(kernel_size=2)

    )

    self.classifier = nn.Sequential(
        nn.Flatten(),
        nn.Linear(256*1*1, 128),
        nn.ReLU(),
        nn.Dropout(p=0.5), # To reduce overfitting
        nn.Linear(128, num_classes)
    )

  def forward(self, x):
    x = self.features(x)
    x = self.classifier(x)
    return x

In [ ]:
def train_one_epoch(model, dataloader, criterion, optimizer, device):
    model.train()  # Set model to training mode
    total_loss = 0
    correct = 0
    total = 0

    for images, labels in dataloader:
        images, labels = images.to(device), labels.to(device)


        outputs = model(images)  # Forward pass
        loss = criterion(outputs, labels)  # Compute loss

        optimizer.zero_grad()  # Reset gradients
        loss.backward()  # Backpropagation
        optimizer.step()  # Update weights

        total_loss += loss.item()

        # Track accuracy
        outputs = torch.softmax(outputs, dim=1)
        predictions = outputs.argmax(dim=1)  # Get class with highest probability
        correct += (predictions == labels).sum().item()
        total += labels.size(0)

    avg_loss = total_loss / len(dataloader)
    accuracy = 100 * correct / total  # Compute accuracy in percentage
    return avg_loss, accuracy

In [ ]:
def validate(model, dataloader, criterion, device):
    model.eval()  # Set model to evaluation mode, you will understand why later
    total_loss = 0 # To check generalization, compare with training loss and detect overfitting
    correct = 0
    total = 0 # Unlike training, we keep track of correct and total to check for accuracy at the end, training doesn't need accuracy. DO NOT GET CONFUSED, adding accuracy to the training is harmless
    # But not a good signal for understanding

    with torch.no_grad():  # Disable gradient calculation
        for images, labels in dataloader:
            images, labels = images.to(device), labels.to(device)

            outputs = model(images)  # Forward pass
            loss = criterion(outputs, labels)  # Compute loss
            total_loss += loss.item()

            # (Optional) Compute accuracy
            predictions = outputs.argmax(dim=1)  # Get class with highest probability
            correct += (predictions == labels).sum().item()
            total += labels.size(0)

    avg_loss = total_loss / len(dataloader)
    accuracy = 100 * correct / total  # Compute accuracy in percentage
    return avg_loss, accuracy

In [ ]:
# Write your code here
# Initialize the model
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = SimpleCNN().to(device)

# Define loss function and optimizer
criterion = nn.CrossEntropyLoss()  # Multi-class Classification loss (Input: Logits, not probabilities)
optimizer = optim.Adam(model.parameters(), lr=0.001, weight_decay=1e-4)  # Adam optimizer
num_epochs = 5 # Number of epochs


# Lists to store metrics
train_losses = []
val_losses = []
train_accuracies = []
val_accuracies = []

# Training process
for epoch in range(num_epochs):
    train_loss, train_accuracy = train_one_epoch(model, train_loader, criterion, optimizer, device)
    val_loss, val_accuracy = validate(model, test_loader, criterion, device)

    # Store metrics
    train_losses.append(train_loss)
    val_losses.append(val_loss)
    train_accuracies.append(train_accuracy)
    val_accuracies.append(val_accuracy)

    print(f"Epoch {epoch+1}/{num_epochs}: "
          f"Train Loss={train_loss:.4f}, Train Accuracy={train_accuracy:.2f}%, "
          f"Val Loss={val_loss:.4f}, Val Accuracy={val_accuracy:.2f}%")


In [ ]:
import matplotlib.pyplot as plt

# Plot loss curve
plt.figure(figsize=(12, 5))
plt.subplot(1, 2, 1)
plt.plot(range(1, num_epochs+1), train_losses, label="Train Loss", marker='o')
plt.plot(range(1, num_epochs+1), val_losses, label="Validation Loss", marker='o')
plt.xlabel("Epochs")
plt.ylabel("Loss")
plt.title("Loss Curve")
plt.legend()

# Plot accuracy curve
plt.subplot(1, 2, 2)
plt.plot(range(1, num_epochs+1), train_accuracies, label="Train Accuracy", marker='o')
plt.plot(range(1, num_epochs+1), val_accuracies, label="Validation Accuracy", marker='o')
plt.xlabel("Epochs")
plt.ylabel("Accuracy (%)")
plt.title("Accuracy Curve")
plt.legend()

plt.show()

# YIPPEEEE